# Churn Prediction Training & Evaluation

**Dataset:** UCI Online Retail II  
**Models:** Logistic Regression (baseline) + Random Forest (primary)  
**Artifacts saved to:** `../app/models/`

## 1. Setup & imports

In [57]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

ARTIFACTS_PATH = "../app/models/"
os.makedirs(ARTIFACTS_PATH, exist_ok=True)

print("Done.")

Done.


## 2. Load data

In [58]:
df = pd.read_excel("online_retail_II.xlsx", sheet_name="Year 2010-2011", engine="openpyxl")

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head()

Rows: 541,910
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 3. Data cleaning

In [59]:
print(f"Rows before cleaning: {len(df):,}")

# Drop rows with no CustomerID, needed for customer history
df = df.dropna(subset=["Customer ID"])

# Drop cancellations (invoices starting with C)
df = df[~df["Invoice"].astype(str).str.startswith("C")]

# Drop rows with non-positive quantity or price
df = df[df["Quantity"] > 0]
df = df[df["Price"] > 0]

# Make sure InvoiceDate is datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Make sure CustomerID is int
df["Customer ID"] = df["Customer ID"].astype(int)

print(f"Rows after cleaning: {len(df):,}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")

Rows before cleaning: 541,910
Rows after cleaning: 397,885
Unique customers: 4,338


## 4. Labelling, Temporal split

**Note for Temporal split:**
The naive approach, which computes recency from the full dataset and labelling churn (ex. if `recency > 90 days`) causes target leakage. Recency directly encodes the label, so the model learns to restate the definition rather than predict anything. 

A temporal split fixes this by separating the past window (what we know) from the future window (what we are predicting):
- **Past window** (before cutoff): RFM features computed here; customer history.
- **Future window** (after cutoff): check whether each customer made a purchase here; basis for if-churned


**Cutoff date:** The dataset ends Dec 9 2011. Using a 90-day outcome window (`Dec 9 - 90 days = Sep 10 2011`), we round to **Sep 1 2011** as a clean cutoff.

**Churn label definition:** A customer is churned if they made zero purchases in the future window (Sep 1 to Dec 9 2011)

In [60]:
CUTOFF_DATE = pd.Timestamp('2011-09-01')

past = df[df['InvoiceDate'] < CUTOFF_DATE].copy()
future = df[df['InvoiceDate'] >= CUTOFF_DATE].copy()

print(f'Cutoff date:          {CUTOFF_DATE.date()}')
print(f'Dataset end date:     {df["InvoiceDate"].max().date()}')
print(f'Outcome window:       {(df["InvoiceDate"].max() - CUTOFF_DATE).days} days')
print(f'')
print(f'Past transactions:    {len(past):,}')
print(f'Future transactions:  {len(future):,}')
print(f'Customers in past:    {past["Customer ID"].nunique():,}')
print(f'Customers in future:  {future["Customer ID"].nunique():,}')

returned_customers = set(future['Customer ID'].unique())

Cutoff date:          2011-09-01
Dataset end date:     2011-12-09
Outcome window:       99 days

Past transactions:    226,467
Future transactions:  171,418
Customers in past:    3,317
Customers in future:  2,973


## 5. Feature engineering (RFM)

| Feature | Why included |
|---|---|
| `recency` | How recently was the customer active |
| `frequency` | Distinguishes loyal repeat buyers from one-time purchasers |
| `monetary` | High-value customers may have different churn patterns |
| `avg_order_value` | Captures basket size independent of visit frequency |
| `total_items` | Volume signal separate from order count |
| `unique_products` | Breadth of engagement with the catalogue |

In [61]:
# Revenue per transaction (past window only)
past['Revenue'] = past['Quantity'] * past['Price']

# Aggregate RFM features per customer from past transactions only
# Recency measured from cutoff date (not snapshot date)
rfm = past.groupby('Customer ID').agg(
    recency=('InvoiceDate', lambda x: (CUTOFF_DATE - x.max()).days),
    frequency=('Invoice', 'nunique'),
    monetary=('Revenue', 'sum'),
    avg_order_value=('Revenue', 'mean'),
    total_items=('Quantity', 'sum'),
    unique_products=('StockCode', 'nunique'),
).reset_index()

# 0 = returned (not churned), 1 = did not return (churned)
rfm['churned'] = (~rfm['Customer ID'].isin(returned_customers)).astype(int)

print(f'Customers with past history: {len(rfm):,}')
print(f'')
print(f'Churn distribution:')
print(rfm['churned'].value_counts())
print(f'')
print(f'Churn rate: {rfm["churned"].mean():.1%}')

rfm.head()

Customers with past history: 3,317

Churn distribution:
churned
0    1952
1    1365
Name: count, dtype: int64

Churn rate: 41.2%


,Customer ID,recency,frequency,monetary,avg_order_value,total_items,unique_products,churned
0,12346,225,1,77183.60,77183.600000,74215,1,1
1,12347,29,5,2790.86,22.506935,1590,82,0
2,12348,148,3,1487.24,53.115714,2124,22,0
3,12350,210,1,334.40,19.670588,197,17,1
4,12352,162,5,1561.81,41.100263,254,26,0


## 6. Data preprocessing

In [62]:
FEATURE_COLUMNS = ['recency', 'frequency', 'monetary', 'avg_order_value', 'total_items', 'unique_products']

X = rfm[FEATURE_COLUMNS]
y = rfm['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train size: {len(X_train):,}')
print(f'Test size:  {len(X_test):,}')
print(f'Features:   {FEATURE_COLUMNS}')

Train size: 2,653
Test size:  664
Features:   ['recency', 'frequency', 'monetary', 'avg_order_value', 'total_items', 'unique_products']


## 7. Model training

In [63]:
# Logistic Regression 
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
print("Logistic Regression trained.")

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
print("Random Forest trained.")

Logistic Regression trained.
Random Forest trained.


## 8. Model evaluation

In [64]:
def evaluate(name, model, X_test, y_test):
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    print(f" {name}")
    print(classification_report(y_test, preds, target_names=["Active", "Churned"]))
    print(f"ROC-AUC: {roc_auc_score(y_test, proba):.4f}")
    return roc_auc_score(y_test, proba)

lr_auc = evaluate("Logistic Regression", lr, X_test_scaled, y_test)
print(f'')
rf_auc = evaluate("Random Forest", rf, X_test_scaled, y_test)

print(f"\nWinner: {'Random Forest' if rf_auc >= lr_auc else 'Logistic Regression'} (AUC {max(rf_auc, lr_auc):.4f})")

 Logistic Regression
              precision    recall  f1-score   support

      Active       0.71      0.69      0.70       391
     Churned       0.57      0.59      0.58       273

    accuracy                           0.65       664
   macro avg       0.64      0.64      0.64       664
weighted avg       0.65      0.65      0.65       664

ROC-AUC: 0.7356

 Random Forest
              precision    recall  f1-score   support

      Active       0.71      0.69      0.70       391
     Churned       0.58      0.60      0.59       273

    accuracy                           0.66       664
   macro avg       0.65      0.65      0.65       664
weighted avg       0.66      0.66      0.66       664

ROC-AUC: 0.7160

Winner: Logistic Regression (AUC 0.7356)


In [65]:
# For Random Forest
importance_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("Feature importances (Random Forest):")
print(importance_df.to_string(index=False))

Feature importances (Random Forest):
        feature  importance
       monetary    0.198089
    total_items    0.195790
        recency    0.182653
avg_order_value    0.174431
unique_products    0.151010
      frequency    0.098028


## 9. Artifacts

In [66]:
best_model = lr # LR won on AUC (0.7356 vs 0.7160)

joblib.dump(best_model, os.path.join(ARTIFACTS_PATH, "churn_model.joblib"))
joblib.dump(scaler, os.path.join(ARTIFACTS_PATH, "scaler.joblib"))
joblib.dump(FEATURE_COLUMNS, os.path.join(ARTIFACTS_PATH, "feature_columns.joblib"))

print("Artifacts saved:")
for f in os.listdir(ARTIFACTS_PATH):
    fpath = os.path.join(ARTIFACTS_PATH, f)
    print(f"  {f} ({os.path.getsize(fpath):,} bytes)")

Artifacts saved:
  churn_model.joblib (911 bytes)
  feature_columns.joblib (99 bytes)
  scaler.joblib (1,095 bytes)
